# Introduction to Generalized Linear Models

What do Linear Regression, Logistic Regression, and Poisson regression have in common?

In each case we 
1. Constrain $(Y \mid X = x)$ to come from some particular family of distributions (normal, Bernoulli, and Poisson respectively).  Different values of $x$ give us different distributions within the same family.
2. We specify a linear predictor $\eta(x) = x \cdot \beta$ 
3. We specify a "link function" $g$ such that $g(\mathbb{E}(Y \mid X = x)) = \eta(x)$ (identity, logit, and log respectively).

Each time we make a specification of a different family of distributions for the target and link function to relate the expected value of the target to the linear predictor, we are specifying a *generalized linear model*.  For technical reasons, we require the family of distributions to be what is called an *overdispersed exponential family of distributions*.

An *overdispersed exponential family of distributions* are families of probability distributions, parameterized by $\boldsymbol\theta$ and $\tau$, whose density functions $f$ (or probability mass function, for the case of a discrete distribution) can be expressed in the form:

$$ 
f_Y(\mathbf{y} \mid \boldsymbol\theta, \tau) = h(\mathbf{y},\tau) \exp \left(\frac{\mathbf{b}(\boldsymbol\theta)^{\rm T}\mathbf{T}(\mathbf{y}) - A(\boldsymbol\theta)} {d(\tau)} \right)
$$

where

- **$T(y)$** – the *sufficient statistic* (possibly vector-valued).  
- **$b(\theta)$** – the *natural parameter function* (the map from model parameter $\theta$ into the canonical parameter space).  
- **$d(\tau)$** – the *dispersion function* (scales the contribution of the sufficient statistic and cumulant).  
- **$h(y,\tau)$** – the *base measure* (depends only on $y$ and dispersion, not on $\theta$).
- **$A(\theta)$** – the *cumulant generating function* or *log-partition function*.  
    - Derived from the rest as a normalizing factor to make the integral equal to $1$.

We will see in math hour that  

$$
\mathbb{E}[T(Y)] = \nabla_\theta A(\theta)
$$

and  

$$
\operatorname{Var}[T(Y)] = d(\tau)\,\nabla^2_\theta A(\theta).
$$

$Y$ is typically one coordinate of $T(Y)$, so the mean and variance of $Y$ are obtained by selecting the corresponding entry of the vector $\nabla_\theta A(\theta)$ and the corresponding diagonal element of the matrix $d(\tau)\,\nabla^2_\theta A(\theta)$.

| Distribution | PDF / PMF | Canonical Link $g(\mu)$ | $T(y)$ | $b(\theta)$ | $A(\theta)$ | $d(\tau)$ | $h(y,\tau)$ | $\mu(\theta)$ | $\operatorname{Var}(Y)$ | Typical Use |
|--------------|-----------|--------------------------|--------|-------------|-------------|-----------|-------------|---------------|-------------------------|-------------|
| Normal (dispersion $\sigma^2$ known) | $\tfrac{1}{\sqrt{2\pi\sigma^2}}\exp\!\big(-\tfrac{(y-\mu)^2}{2\sigma^2}\big)$ | $g(\mu)=\mu$ | $y$ | $\theta$ | $\tfrac{1}{2}\theta^2$ | $\sigma^2$ | $\tfrac{1}{\sqrt{2\pi\sigma^2}}\exp\!\big(-\tfrac{y^2}{2\sigma^2}\big)$ | $\mu=\theta$ | $\sigma^2$ | Real-valued response |
| Exponential | $\lambda e^{-\lambda y}$ | $g(\mu)=-1/\mu$ | $y$ | $\theta$ | $-\ln(-\theta)$ | $1$ | $\mathbf{1}_{\{y>0\}}$ | $\mu=-1/\theta$ | $\mu^2$ | Positive continuous (scale) |
| Poisson | $\tfrac{\mu^y e^{-\mu}}{y!}$ | $g(\mu)=\ln\mu$ | $y$ | $\theta$ | $e^\theta$ | $1$ | $1/y!$ | $\mu=e^\theta$ | $\mu$ | Count data |
| Bernoulli | $p^y(1-p)^{1-y}$ | $g(\mu)=\ln\tfrac{\mu}{1-\mu}$ | $y$ | $\theta$ | $\ln(1+e^\theta)$ | $1$ | $1$ | $\mu=\tfrac{e^\theta}{1+e^\theta}$ | $\mu(1-\mu)$ | Binary response |
| Binomial ($n$) | $\binom{n}{y}p^y(1-p)^{n-y}$ | $g(\mu)=\ln\tfrac{\mu/n}{1-\mu/n}$ | $y$ | $\theta$ | $n\ln(1+e^\theta)$ | $1$ | $\binom{n}{y}$ | $\mu=n\cdot \tfrac{e^\theta}{1+e^\theta}$ | $\mu\!\left(1-\tfrac{\mu}{n}\right)$ | Number of successes |
| Normal (unknown $\sigma^2$; 2-param EF) | $\tfrac{1}{\sqrt{2\pi\sigma^2}}\exp\!\big(-\tfrac{(y-\mu)^2}{2\sigma^2}\big)$ | $g(\mu)=\mu$ | $(y,\,y^2)$ | $(\theta_1,\theta_2)$ | $-\dfrac{\theta_1^2}{4\theta_2}+\tfrac{1}{2}\ln\!\big(-\tfrac{\pi}{\theta_2}\big)$ | $1$ | $1$ | $\mu=-\dfrac{\theta_1}{2\theta_2}$ | $-\dfrac{1}{2\theta_2}$ | General Gaussian modeling |
| Gamma (shape–rate; 2-param EF) | $\dfrac{\beta^\alpha}{\Gamma(\alpha)}\,y^{\alpha-1}e^{-\beta y}$ | $g(\mu)=1/\mu$ (if shape fixed) | $(\ln y,\,y)$ | $(\theta_1,\theta_2)$ | $\ln\Gamma(\theta_1+1)-( \theta_1+1)\ln(-\theta_2)$ | $1$ | $1$ | $\mu=\dfrac{\theta_1+1}{-\theta_2}$ | $\dfrac{\theta_1+1}{\theta_2^{2}}$ | Positive continuous |
| Inverse Gaussian (dispersion $\phi$) | $\big(2\pi\phi y^3\big)^{-1/2}\exp\!\Big(-\tfrac{(y-\mu)^2}{2\phi\,\mu^2 y}\Big)$ | $g(\mu)=-\tfrac{1}{2\mu^2}$ | $(y,\,1/y)$ | $(\theta_1,\theta_2)$ | $-\sqrt{-2\theta_1\theta_2}$ | $\phi$ | $(2\pi\phi y^3)^{-1/2}$ | $\mu=\sqrt{-\theta_2/\theta_1}$ | $\phi\,\mu^3$ | Positive continuous with skew |


### GLMs: quick mapping and caveats

- **Normal (identity link)** ⇒ ordinary least squares (OLS) linear regression.  
- **Poisson (log link)** ⇒ Poisson regression.  
- **Bernoulli (logit link)** ⇒ logistic regression.  
- Other rows correspond to other GLMs used when the outcome’s mean–variance relationship matches that family.

### Optimization facts (what we’ll show in Math Hour)

For a GLM with design matrix $X$ and linear predictor $\eta=X\beta$:

- Negative log-likelihood is **convex in $\beta$ for canonical links** (and more generally convex in $\eta$ under standard EF regularity and full-rank $X$).  
- Gradient:
  $$
  \nabla_\beta \ell(\beta)=X^\top(\mu - y),
  $$
  where $\mu=g^{-1}(\eta)$.
- Hessian:
  $$
  \nabla^2_\beta \ell(\beta)=X^\top W X,
  $$
  with $W=\operatorname{diag}\!\big(w_i\big)$ and
  $$
  w_i=\frac{1}{d(\tau)}\,\frac{1}{\left(g'(\mu_i)\right)^2}\,V(\mu_i).
  $$
  For canonical links this simplifies (e.g., Poisson: $w_i=\mu_i/d(\tau)$; Bernoulli: $w_i=\mu_i(1-\mu_i)/d(\tau)$).

### Study guidance

There are entire textbooks on GLMs. If you plan to use them routinely, invest beyond this bootcamp. We will not cover full inference for parameters or specification tests, and model choice (including non-canonical links) is a substantive modeling decision.
